In [5]:
import MDAnalysis as mda
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ============================================================
# INPUT FILES
# ============================================================

systems = {
    "BDQ": [
        (
            "/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep1.tpr",
            "/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep1.xtc",
            "BDQ_rep1"
        ),
        (
            "/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep2.tpr",
            "/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep2.xtc",
            "BDQ_rep2"
        ),
        (
            "/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep3.tpr",
            "/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep3.xtc",
            "BDQ_rep3"
        ),
    ],

    "BDN": [
        (
            "/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep1.tpr",
            "/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep1.xtc",
            "BDN_rep1"
        ),
        (
            "/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep2.tpr",
            "/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep2.xtc",
            "BDN_rep2"
        ),
        (
            "/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep3.tpr",
            "/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep3.xtc",
            "BDN_rep3"
        ),
    ]
}

# ============================================================
# DISPLAY NAMES
# ============================================================

display_names = {
    "BDQ": "BDP",
    "BDN": "BDN"
}

# ============================================================
# ATOM SELECTIONS
# ============================================================

def get_dihedral_selection(resname):
    """
    Return atom selection for dihedral calculation.
    """

    return (
        f"resname {resname} and "
        "(name C11 or name C08 or name C06 or name C09)"
    )


def get_full_ligand_selection(resname):
    """
    Return full ligand selection.
    """

    return f"resname {resname}"


# Membrane selection
membrane_selection = "resname POPC POPS POPE"

# ============================================================
# MAIN ANALYSIS FUNCTION
# ============================================================

def dihedral_split_by_distance(
    tpr,
    xtc,
    resname,
    cutoff_A=20.0
):

    # --------------------------------------------------------
    # LOAD TRAJECTORY
    # --------------------------------------------------------

    u = mda.Universe(tpr, xtc)

    print("\n================================================")
    print(f"Processing system: {resname}")
    print("================================================")

    # --------------------------------------------------------
    # ATOM GROUPS
    # --------------------------------------------------------

    dih_atoms = u.select_atoms(
        get_dihedral_selection(resname)
    )

    ligand_full = u.select_atoms(
        get_full_ligand_selection(resname)
    )

    membrane = u.select_atoms(
        membrane_selection
    )

    # --------------------------------------------------------
    # SAFETY CHECKS
    # --------------------------------------------------------

    if len(dih_atoms) != 4:
        raise ValueError(
            f"Dihedral selection returned {len(dih_atoms)} atoms "
            f"instead of 4 for {resname}"
        )

    if len(ligand_full) == 0:
        raise ValueError(
            f"No ligand atoms found for {resname}"
        )

    if len(membrane) == 0:
        raise ValueError(
            "No membrane atoms found"
        )

    # --------------------------------------------------------
    # INFO
    # --------------------------------------------------------

    n_frames = len(u.trajectory)

    print(f"Frames: {n_frames}")
    print(f"Dihedral atoms: {len(dih_atoms)}")
    print(f"Ligand atoms: {len(ligand_full)}")
    print(f"Membrane atoms: {len(membrane)}")

    # --------------------------------------------------------
    # STORAGE
    # --------------------------------------------------------

    angles = []
    dz_values = []

    # --------------------------------------------------------
    # TRAJECTORY LOOP
    # --------------------------------------------------------

    for i, ts in enumerate(u.trajectory):

        if i % 200 == 0:
            print(f"Frame {i}/{n_frames}")

        # ----------------------------------------------------
        # COM DISTANCE
        # ----------------------------------------------------

        ligand_z = ligand_full.center_of_mass()[2]
        membrane_z = membrane.center_of_mass()[2]

        dz = ligand_z - membrane_z

        # ----------------------------------------------------
        # DIHEDRAL
        # ----------------------------------------------------

        angle = dih_atoms.dihedral.value()

        angles.append(angle)
        dz_values.append(dz)

    # --------------------------------------------------------
    # CONVERT TO NUMPY
    # --------------------------------------------------------

    angles = np.array(angles)
    dz_values = np.array(dz_values)

    # --------------------------------------------------------
    # SPLIT ENVIRONMENTS
    # --------------------------------------------------------

    membrane_phase = np.abs(dz_values) <= cutoff_A
    aqueous_phase = np.abs(dz_values) > cutoff_A

    print(f"Membrane frames: {np.sum(membrane_phase)}")
    print(f"Aqueous frames: {np.sum(aqueous_phase)}")

    return (
        angles[membrane_phase],
        angles[aqueous_phase]
    )

# ============================================================
# ANALYSIS LOOP
# ============================================================

rows = []

for group, replicas in systems.items():

    print("\n################################################")
    print(f"GROUP: {group}")
    print("################################################")

    for tpr, xtc, repname in replicas:

        membrane_angles, aqueous_angles = (
            dihedral_split_by_distance(
                tpr=tpr,
                xtc=xtc,
                resname=group,
                cutoff_A=20.0
            )
        )

        # ----------------------------------------------------
        # MEMBRANE DATA
        # ----------------------------------------------------

        rows.append(
            pd.DataFrame({
                "angle": membrane_angles,
                "group": display_names[group],
                "environment": "membrane",
                "replica": repname
            })
        )

        # ----------------------------------------------------
        # AQUEOUS DATA
        # ----------------------------------------------------

        rows.append(
            pd.DataFrame({
                "angle": aqueous_angles,
                "group": display_names[group],
                "environment": "aqueous",
                "replica": repname
            })
        )

# ============================================================
# FINAL DATAFRAME
# ============================================================

df = pd.concat(rows, ignore_index=True)

print("\nData preview:")
print(df.head())

# ============================================================
# PLOTTING
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5),
    sharey=True
)

palette = {
    "BDN": "blue",
    "BDP": "orange"
}

# ============================================================
# MEMBRANE
# ============================================================

sns.histplot(
    data=df[df["environment"] == "membrane"],
    x="angle",
    hue="group",
    bins=60,
    stat="probability",
    element="step",
    fill=True,
    alpha=0.4,
    palette=palette,
    ax=axes[0]
)

axes[0].set_title("Membrane environment")
axes[0].set_xlabel("Dihedral angle (degrees)")
axes[0].set_ylabel("Probability")

# ============================================================
# AQUEOUS
# ============================================================

sns.histplot(
    data=df[df["environment"] == "aqueous"],
    x="angle",
    hue="group",
    bins=60,
    stat="probability",
    element="step",
    fill=True,
    alpha=0.4,
    palette=palette,
    ax=axes[1]
)

axes[1].set_title("Aqueous environment")
axes[1].set_xlabel("Dihedral angle (degrees)")

# ============================================================
# FINALIZE
# ============================================================

plt.tight_layout()

plt.savefig(
    "dih11_8_6_9.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nFinished successfully.")



################################################
GROUP: BDQ
################################################

Processing system: BDQ
Frames: 5001
Dihedral atoms: 4
Ligand atoms: 69
Membrane atoms: 24000
Frame 0/5001
Frame 200/5001
Frame 400/5001
Frame 600/5001
Frame 800/5001
Frame 1000/5001
Frame 1200/5001
Frame 1400/5001
Frame 1600/5001
Frame 1800/5001
Frame 2000/5001
Frame 2200/5001
Frame 2400/5001
Frame 2600/5001
Frame 2800/5001
Frame 3000/5001
Frame 3200/5001
Frame 3400/5001
Frame 3600/5001
Frame 3800/5001
Frame 4000/5001
Frame 4200/5001
Frame 4400/5001
Frame 4600/5001
Frame 4800/5001
Frame 5000/5001
Membrane frames: 4249
Aqueous frames: 752

Processing system: BDQ
Frames: 5001
Dihedral atoms: 4
Ligand atoms: 69
Membrane atoms: 24000
Frame 0/5001
Frame 200/5001
Frame 400/5001
Frame 600/5001
Frame 800/5001
Frame 1000/5001
Frame 1200/5001
Frame 1400/5001
Frame 1600/5001
Frame 1800/5001
Frame 2000/5001
Frame 2200/5001
Frame 2400/5001
Frame 2600/5001
Frame 2800/5001
Frame 3000/5001
Fr

In [7]:
import MDAnalysis as mda
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# -------------------------
# INPUT
# -------------------------
systems = {
    "BDQ": [
        (
            "/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep1.tpr",
            "/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep1.xtc",
            "BDQ_rep1"
        ),
        (
            "/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep2.tpr",
            "/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep2.xtc",
            "BDQ_rep2"
        ),
        (
            "/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep3.tpr",
            "/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep3.xtc",
            "BDQ_rep3"
        ),
    ],

    "BDN": [
        (
            "/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep1.tpr",
            "/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep1.xtc",
            "BDN_rep1"
        ),
        (
            "/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep2.tpr",
            "/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep2.xtc",
            "BDN_rep2"
        ),
        (
            "/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep3.tpr",
            "/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep3.xtc",
            "BDN_rep3"
        ),
    ]
}

# -------------------------
# DISPLAY NAMES (THESIS)
# -------------------------
display_names = {
    "BDQ": "BDP",
    "BDN": "BDN"
}

# -------------------------
# DIHEDRAL SELECTION
# -------------------------
def get_dihedral_selection(resname):

    if resname == "BDQ":
        return (
            "resname BDQ and "
            "(name C08 or name C06 or name C07 or name C12)"
        )

    elif resname == "BDN":
        return (
            "resname BDN and "
            "(name C08 or name C06 or name C07 or name C12)"
        )

    else:
        raise ValueError(f"Unknown residue name: {resname}")

# -------------------------
# MEMBRANE SELECTION
# -------------------------
membrane_selection = "resname POPC POPS POPE"

# -------------------------
# MAIN FUNCTION
# -------------------------
def dihedral_split_by_distance(
    tpr,
    xtc,
    resname,
    cutoff_A=20.0
):

    u = mda.Universe(tpr, xtc)

    print("\n==============================")
    print(f"System: {resname} → {display_names[resname]}")

    # Full ligand for COM calculation
    ligand_full = u.select_atoms(f"resname {resname}")

    # Only 4 atoms for dihedral angle
    ligand_dihedral = u.select_atoms(
        get_dihedral_selection(resname)
    )

    # Membrane
    membrane = u.select_atoms(membrane_selection)

    n_frames = len(u.trajectory)

    print(f"Total frames: {n_frames}")
    print(f"Full ligand atoms: {len(ligand_full)}")
    print(f"Dihedral atoms: {len(ligand_dihedral)}")
    print(f"Membrane atoms: {len(membrane)}")

    angles = []
    dz_values = []

    for i, ts in enumerate(u.trajectory):

        if i % 200 == 0:
            print(f"Processing frame {i}/{n_frames}")

        # COM distance using FULL ligand
        dz = (
            ligand_full.center_of_mass()[2]
            - membrane.center_of_mass()[2]
        )

        # Dihedral angle using ONLY 4 atoms
        angle = ligand_dihedral.dihedral.value()

        angles.append(angle)
        dz_values.append(dz)

    angles = np.array(angles)
    dz_values = np.array(dz_values)

    # -------------------------
    # PHASE SPLIT
    # -------------------------
    membrane_phase = np.abs(dz_values) <= cutoff_A
    bulk_phase = np.abs(dz_values) > cutoff_A

    print(f"Membrane phase frames: {np.sum(membrane_phase)}")
    print(f"Bulk phase frames: {np.sum(bulk_phase)}")

    return (
        angles[membrane_phase],
        angles[bulk_phase]
    )

# -------------------------
# DATA COLLECTION
# -------------------------
rows = []

for group, reps in systems.items():

    for tpr, xtc, repname in reps:

        membrane_angles, bulk_angles = (
            dihedral_split_by_distance(
                tpr,
                xtc,
                group,
                cutoff_A=20.0
            )
        )

        rows.append(
            pd.DataFrame({
                "angle": membrane_angles,
                "group": display_names[group],
                "phase": "membrane",
                "replica": repname
            })
        )

        rows.append(
            pd.DataFrame({
                "angle": bulk_angles,
                "group": display_names[group],
                "phase": "bulk",
                "replica": repname
            })
        )

df = pd.concat(rows, ignore_index=True)

# -------------------------
# PLOT
# -------------------------
fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5),
    sharey=True
)

palette = {
    "BDN": "blue",
    "BDP": "orange"
}

sns.histplot(
    data=df[df["phase"] == "membrane"],
    x="angle",
    hue="group",
    bins=60,
    stat="probability",
    element="step",
    fill=True,
    alpha=0.4,
    palette=palette,
    ax=axes[0]
)

axes[0].set_title("Membrane phase")
axes[0].set_xlabel("Dihedral angle (°)")

sns.histplot(
    data=df[df["phase"] == "bulk"],
    x="angle",
    hue="group",
    bins=60,
    stat="probability",
    element="step",
    fill=True,
    alpha=0.4,
    palette=palette,
    ax=axes[1]
)

axes[1].set_title("Bulk phase")
axes[1].set_xlabel("Dihedral angle (°)")

plt.tight_layout()
plt.savefig("dih8_6_7_12.png", dpi=300)
plt.show()


System: BDQ → BDP
Total frames: 5001
Full ligand atoms: 69
Dihedral atoms: 4
Membrane atoms: 24000
Processing frame 0/5001
Processing frame 200/5001
Processing frame 400/5001
Processing frame 600/5001
Processing frame 800/5001
Processing frame 1000/5001
Processing frame 1200/5001
Processing frame 1400/5001
Processing frame 1600/5001
Processing frame 1800/5001
Processing frame 2000/5001
Processing frame 2200/5001
Processing frame 2400/5001
Processing frame 2600/5001
Processing frame 2800/5001
Processing frame 3000/5001
Processing frame 3200/5001
Processing frame 3400/5001
Processing frame 3600/5001
Processing frame 3800/5001
Processing frame 4000/5001
Processing frame 4200/5001
Processing frame 4400/5001
Processing frame 4600/5001
Processing frame 4800/5001
Processing frame 5000/5001
Membrane phase frames: 4249
Bulk phase frames: 752

System: BDQ → BDP
Total frames: 5001
Full ligand atoms: 69
Dihedral atoms: 4
Membrane atoms: 24000
Processing frame 0/5001
Processing frame 200/5001
Proc

In [8]:
import MDAnalysis as mda
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# -------------------------
# INPUT
# -------------------------
systems = {
    "BDQ": [
        (
            "/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep1.tpr",
            "/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep1.xtc",
            "BDQ_rep1"
        ),
        (
            "/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep2.tpr",
            "/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep2.xtc",
            "BDQ_rep2"
        ),
        (
            "/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep3.tpr",
            "/scratch/gent/vo/000/gvo00003/vsc48847/mem_bdq2/memb_bdq2/Equil/step7_production_rep3.xtc",
            "BDQ_rep3"
        ),
    ],

    "BDN": [
        (
            "/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep1.tpr",
            "/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep1.xtc",
            "BDN_rep1"
        ),
        (
            "/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep2.tpr",
            "/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep2.xtc",
            "BDN_rep2"
        ),
        (
            "/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep3.tpr",
            "/scratch/gent/vo/000/gvo00003/vsc48847/memb_bdq/Equil/step7_production_rep3.xtc",
            "BDN_rep3"
        ),
    ]
}

# -------------------------
# DISPLAY NAMES (THESIS)
# -------------------------
display_names = {
    "BDQ": "BDP",
    "BDN": "BDN"
}

# -------------------------
# DIHEDRAL SELECTION
# -------------------------
def get_dihedral_selection(resname):

    if resname == "BDQ":
        return (
            "resname BDQ and "
            "(name C11 or name C08 or name C06 or name C10)"
        )

    elif resname == "BDN":
        return (
            "resname BDN and "
            "(name C11 or name C08 or name C06 or name C10)"
        )

    else:
        raise ValueError(f"Unknown residue name: {resname}")

# -------------------------
# MEMBRANE SELECTION
# -------------------------
membrane_selection = "resname POPC POPS POPE"

# -------------------------
# MAIN FUNCTION
# -------------------------
def dihedral_split_by_distance(
    tpr,
    xtc,
    resname,
    cutoff_A=20.0
):

    u = mda.Universe(tpr, xtc)

    print("\n==============================")
    print(f"System: {resname} → {display_names[resname]}")

    # Full ligand for COM calculation
    ligand_full = u.select_atoms(f"resname {resname}")

    # Only 4 atoms for dihedral angle
    ligand_dihedral = u.select_atoms(
        get_dihedral_selection(resname)
    )

    # Membrane
    membrane = u.select_atoms(membrane_selection)

    n_frames = len(u.trajectory)

    print(f"Total frames: {n_frames}")
    print(f"Full ligand atoms: {len(ligand_full)}")
    print(f"Dihedral atoms: {len(ligand_dihedral)}")
    print(f"Membrane atoms: {len(membrane)}")

    angles = []
    dz_values = []

    for i, ts in enumerate(u.trajectory):

        if i % 200 == 0:
            print(f"Processing frame {i}/{n_frames}")

        # COM distance using FULL ligand
        dz = (
            ligand_full.center_of_mass()[2]
            - membrane.center_of_mass()[2]
        )

        # Dihedral angle using ONLY 4 atoms
        angle = ligand_dihedral.dihedral.value()

        angles.append(angle)
        dz_values.append(dz)

    angles = np.array(angles)
    dz_values = np.array(dz_values)

    # -------------------------
    # PHASE SPLIT
    # -------------------------
    membrane_phase = np.abs(dz_values) <= cutoff_A
    bulk_phase = np.abs(dz_values) > cutoff_A

    print(f"Membrane phase frames: {np.sum(membrane_phase)}")
    print(f"Bulk phase frames: {np.sum(bulk_phase)}")

    return (
        angles[membrane_phase],
        angles[bulk_phase]
    )

# -------------------------
# DATA COLLECTION
# -------------------------
rows = []

for group, reps in systems.items():

    for tpr, xtc, repname in reps:

        membrane_angles, bulk_angles = (
            dihedral_split_by_distance(
                tpr,
                xtc,
                group,
                cutoff_A=20.0
            )
        )

        rows.append(
            pd.DataFrame({
                "angle": membrane_angles,
                "group": display_names[group],
                "phase": "membrane",
                "replica": repname
            })
        )

        rows.append(
            pd.DataFrame({
                "angle": bulk_angles,
                "group": display_names[group],
                "phase": "bulk",
                "replica": repname
            })
        )

df = pd.concat(rows, ignore_index=True)

# -------------------------
# PLOT
# -------------------------
fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5),
    sharey=True
)

palette = {
    "BDN": "blue",
    "BDP": "orange"
}

sns.histplot(
    data=df[df["phase"] == "membrane"],
    x="angle",
    hue="group",
    bins=60,
    stat="probability",
    element="step",
    fill=True,
    alpha=0.4,
    palette=palette,
    ax=axes[0]
)

axes[0].set_title("Membrane phase")
axes[0].set_xlabel("Dihedral angle (°)")

sns.histplot(
    data=df[df["phase"] == "bulk"],
    x="angle",
    hue="group",
    bins=60,
    stat="probability",
    element="step",
    fill=True,
    alpha=0.4,
    palette=palette,
    ax=axes[1]
)

axes[1].set_title("Bulk phase")
axes[1].set_xlabel("Dihedral angle (°)")

plt.tight_layout()
plt.savefig("dih11_8_6_10.png", dpi=300)
plt.show()


System: BDQ → BDP
Total frames: 5001
Full ligand atoms: 69
Dihedral atoms: 4
Membrane atoms: 24000
Processing frame 0/5001
Processing frame 200/5001
Processing frame 400/5001
Processing frame 600/5001
Processing frame 800/5001
Processing frame 1000/5001
Processing frame 1200/5001
Processing frame 1400/5001
Processing frame 1600/5001
Processing frame 1800/5001
Processing frame 2000/5001
Processing frame 2200/5001
Processing frame 2400/5001
Processing frame 2600/5001
Processing frame 2800/5001
Processing frame 3000/5001
Processing frame 3200/5001
Processing frame 3400/5001
Processing frame 3600/5001
Processing frame 3800/5001
Processing frame 4000/5001
Processing frame 4200/5001
Processing frame 4400/5001
Processing frame 4600/5001
Processing frame 4800/5001
Processing frame 5000/5001
Membrane phase frames: 4249
Bulk phase frames: 752

System: BDQ → BDP
Total frames: 5001
Full ligand atoms: 69
Dihedral atoms: 4
Membrane atoms: 24000
Processing frame 0/5001
Processing frame 200/5001
Proc